# 📱 Documentación del Frontend - Sistema de Gestión de Inventarios

## Índice
1. [Introducción](#introduccion)
2. [Arquitectura del Frontend](#arquitectura)
3. [Componentes Principales](#componentes)
4. [Navegación y Rutas](#navegacion)
5. [Funcionalidades Implementadas](#funcionalidades)
6. [Capturas de Pantalla](#capturas)

---

## 1. Introducción {#introduccion}

Este documento describe la implementación del frontend de la aplicación de **Gestión de Inventarios con Predicción de Stock basada en IA**. 

La aplicación está desarrollada con:
- **React + TypeScript**
- **Vite** como bundler
- **TailwindCSS** para estilos
- **Hooks personalizados** para la gestión de estado

## 2. Arquitectura del Frontend {#arquitectura}

### Estructura de Directorios

```
src/
├── pages/           # Páginas principales de la aplicación
│   ├── Productos.tsx    # Vista de gestión de inventario
│   └── Chat.tsx         # Vista de chat con IA
├── hooks/           # Custom hooks para lógica reutilizable
│   └── useInventario.ts # Hook principal para API calls
├── data_acces/      # Capa de acceso a datos
│   ├── inventario_acces.ts
│   └── db/schema/
├── layout/          # Componentes de layout
├── assets/          # Recursos estáticos
└── App.tsx          # Componente raíz con navegación
```

### Patrón de Diseño

La aplicación sigue el patrón de **Container/Presentational Components**:
- **Containers**: Manejan la lógica y el estado (`Productos.tsx`, `Chat.tsx`)
- **Custom Hooks**: Centralizan las llamadas API y estado compartido (`useInventario.ts`)
- **Presentational**: Componentes visuales sin lógica de negocio

## 3. Componentes Principales {#componentes}

### 3.1 App.tsx - Componente Raíz

El componente principal que gestiona la navegación entre vistas.

In [ ]:
# Código simplificado de App.tsx
"""
function App() {
  const [currentView, setCurrentView] = useState<"productos" | "chat">("productos");

  return (
    <div className="min-h-screen bg-gray-100">
      <nav className="bg-white shadow-md">
        <div className="container mx-auto px-6 py-4">
          <h1>Sistema de Gestión de Inventarios</h1>
          <div className="flex gap-2">
            <button onClick={() => setCurrentView("productos")}>
              📦 Productos
            </button>
            <button onClick={() => setCurrentView("chat")}>
              💬 Chat
            </button>
          </div>
        </div>
      </nav>
      
      {currentView === "productos" ? <Productos /> : <Chat />}
    </div>
  );
}
"""

print("✓ App.tsx gestiona la navegación entre 'Productos' y 'Chat'")

### 📸 Captura de Pantalla: Barra de Navegación

**Inserta aquí una captura de pantalla mostrando la barra de navegación con los botones "Productos" y "Chat"**

---

### 3.2 Productos.tsx - Gestión de Inventario

Este componente es el núcleo de la gestión de inventarios. Incluye:

#### Características principales:
1. **Tabla de productos paginada** con información detallada
2. **Búsqueda de productos** por código, nombre o descripción
3. **Predicción de stock individual** para cada producto
4. **Análisis de restock** masivo
5. **Reentrenamiento del modelo** con datos CSV

#### Interfaces TypeScript utilizadas:

In [ ]:
# Interfaces principales del componente Productos
"""
interface PredictionResult {
  productId: number;
  stock_predicho: number;
  target_date: string;
  product_info?: {
    id: number;
    codigo?: string;
    item?: string;
    current_stock: number;
  };
  analysis?: string | null;  // Análisis generado por IA
}

interface RestockItem {
  product_id: number;
  codigo?: string;
  item?: string;
  current_stock: number;
  predicted_stock: number;
  safety_threshold: number;
  restock_amount: number;
  predicted_shortage: boolean;
  unit_cost?: string;
  total_cost: number;
  target_date: string;
}

interface RestockResponse {
  date: string;
  safety_threshold: number;
  page: number;
  limit: number;
  total_pages: number;
  total_products_in_db: number;
  products_analyzed_in_page: number;
  products_needing_restock: number;
  total_restock_cost: number;
  restock_list: RestockItem[];
  analysis?: string | null;  // Análisis profesional generado por IA
}
"""

print("✓ Interfaces TypeScript para manejar predicciones y restock")

### 📸 Captura de Pantalla: Vista Principal de Productos

**Inserta aquí una captura de pantalla mostrando:**
- La tabla de productos con columnas: ID, Código, Item, Stock, Precios
- Barra de búsqueda
- Botones "Análisis de Restock" y "Reentrenar Modelo"
- Paginación en la parte inferior

---

## 4. Funcionalidades Implementadas {#funcionalidades}

### 4.1 Predicción de Stock Individual

Permite predecir el stock futuro de un producto específico en una fecha determinada.

In [ ]:
# Flujo de predicción de stock individual
"""
1. Usuario hace clic en "Predecir Stock" para un producto
2. Se abre un modal con:
   - ID del producto (disabled)
   - Selector de fecha
   - Botón "Obtener Predicción"
3. Al hacer clic, se llama a la API:
   POST /api/predict
   Body: { product_id: number, date: string }
4. La API devuelve:
   {
     product_id: number,
     stock_predicho: number,
     target_date: string,
     product_info: {...},
     analysis: string  // Análisis generado por IA
   }
5. Se muestra en el modal:
   - Información del producto
   - Stock actual vs Stock predicho
   - Fecha objetivo
   - Análisis profesional con recomendaciones
"""

print("✓ Predicción individual con análisis de IA integrado")

### 📸 Captura de Pantalla: Modal de Predicción de Stock

**Inserta aquí una captura de pantalla mostrando:**
- Modal de predicción abierto
- Información del producto seleccionado
- Selector de fecha
- Resultado de la predicción con:
  - Stock actual
  - Stock predicho
  - Análisis de IA con recomendaciones

---

### 4.2 Análisis de Restock Masivo

Analiza múltiples productos simultáneamente para determinar cuáles necesitan reabastecimiento.

In [ ]:
# Flujo de análisis de restock masivo
"""
1. Usuario hace clic en "Análisis de Restock"
2. Se abre un modal con:
   - Selector de fecha para el análisis
   - Campo para umbral de seguridad (default: 10 unidades)
   - Botón "Iniciar Análisis"
3. Al hacer clic, se llama a la API:
   POST /api/check-restock
   Body: {
     date: string,
     threshold: number,
     page: number,
     limit: number
   }
4. La API analiza TODOS los productos y devuelve:
   {
     products_analyzed_in_page: number,
     products_needing_restock: number,
     total_restock_cost: number,
     restock_list: [...],  // TODOS los productos con predicciones
     analysis: string      // Análisis profesional por IA
   }
5. Se muestra en el modal:
   - Panel con análisis de IA (evaluación, recomendaciones, alertas)
   - Estadísticas generales (productos analizados, que necesitan restock, inversión)
   - Tabla con TODOS los productos ordenados por necesidad de restock
   - Paginación para navegar entre páginas de productos
"""

print("✓ Análisis masivo con reporte completo generado por IA")

### 📸 Captura de Pantalla: Modal de Análisis de Restock

**Inserta aquí una captura de pantalla mostrando:**
- Modal de análisis de restock abierto
- Panel destacado con el análisis profesional de IA
- Estadísticas: productos analizados, necesitan restock, inversión total
- Tabla completa con todos los productos y sus predicciones
- Información detallada: código, producto, stock actual, stock predicho, cantidad a reabastecer, costos

---

### 4.3 Reentrenamiento del Modelo

Permite actualizar el modelo de Machine Learning con nuevos datos históricos.

In [ ]:
# Flujo de reentrenamiento del modelo
"""
1. Usuario hace clic en "Reentrenar Modelo"
2. Se abre un modal con:
   - Input de archivo CSV
   - Especificación de campos requeridos
   - Botón "Iniciar Reentrenamiento"
3. El CSV debe contener:
   - product_id, created_at, salida, quantity_on_hand
   - unit_cost, dia_semana, mes, fin_semana, feriado
4. El frontend:
   - Lee el archivo CSV
   - Parsea los datos
   - Los envía a la API en lotes si es necesario
5. La API maneja errores 413 (Payload Too Large) automáticamente:
   - Divide el lote en mitades
   - Procesa recursivamente
6. Se muestra mensaje de éxito o error

Ventajas:
- Manejo automático de archivos grandes
- División inteligente de lotes
- Sin límite de tamaño de archivo
"""

print("✓ Reentrenamiento con manejo automático de lotes grandes")

### 📸 Captura de Pantalla: Modal de Reentrenamiento

**Inserta aquí una captura de pantalla mostrando:**
- Modal de reentrenamiento abierto
- Selector de archivo CSV
- Descripción de campos requeridos
- Mensaje de éxito después del reentrenamiento

---

### 3.3 Chat.tsx - Asistente Virtual

Componente de chat interactivo para consultas generales con el sistema.

In [ ]:
# Características del Chat
"""
Estructura del componente Chat:

1. Estado de mensajes:
   interface Message {
     id: number;
     text: string;
     isUser: boolean;
     timestamp: Date;
   }

2. Funcionalidades:
   - Historial completo de conversación
   - Mensaje inicial de bienvenida automático
   - Scroll automático al último mensaje
   - Indicador visual "escribiendo..." mientras la IA responde
   - Timestamps en cada mensaje

3. Diseño UI:
   - Header con gradiente azul
   - Área de mensajes scrollable
   - Burbujas diferenciadas (usuario: azul, bot: blanco)
   - Input fijo en la parte inferior
   - Botón de envío deshabilitado mientras procesa

4. Comunicación con API:
   POST /api/chat
   Body: { pregunta: string }
   Response: { pregunta: string, respuesta: string }
"""

print("✓ Chat interactivo con IA para consultas generales")

### 📸 Captura de Pantalla: Vista de Chat

**Inserta aquí una captura de pantalla mostrando:**
- Interfaz de chat con mensajes del usuario y del bot
- Header con título "Chat con el Sistema"
- Burbujas de mensaje diferenciadas por color
- Área de input en la parte inferior
- Timestamps en los mensajes
- Indicador "escribiendo..." (si es posible capturarlo)

---

## 5. Hook useInventario - Gestión de Estado {#hook}

El custom hook `useInventario` centraliza toda la lógica de comunicación con el backend y gestión de estado.

In [ ]:
# Estructura del hook useInventario
"""
Estados gestionados:
- inventario: Inventario[]          # Lista de productos
- loading: boolean                  # Carga general
- error: string | null              # Errores generales
- page: number                      # Página actual
- totalPages: number                # Total de páginas
- searchTerm: string                # Término de búsqueda activo
- predicting: boolean               # Cargando predicción
- predictionError: string | null    # Error de predicción
- retraining: boolean               # Cargando reentrenamiento
- retrainError: string | null       # Error de reentrenamiento
- checkingRestock: boolean          # Cargando análisis restock
- restockError: string | null       # Error de restock
- chatting: boolean                 # Cargando respuesta chat
- chatError: string | null          # Error de chat

Funciones exportadas:
- nextPage(): void                  # Ir a página siguiente
- prevPage(): void                  # Ir a página anterior
- goToPage(page): void              # Ir a página específica
- refresh(): void                   # Recargar datos
- search(term): void                # Buscar productos
- clearSearch(): void               # Limpiar búsqueda
- predictStock(id, date): Promise   # Predecir stock individual
- retrainModel(data): Promise       # Reentrenar modelo
- checkRestock(date, ...): Promise  # Análisis de restock
- sendChatMessage(pregunta): Promise # Enviar mensaje al chat

Endpoints consumidos:
- GET  /api/inventario              # Listar productos paginados
- GET  /api/inventario/search/:term # Buscar productos
- POST /api/predict                 # Predicción individual
- POST /api/retrain                 # Reentrenamiento
- POST /api/check-restock           # Análisis de restock
- POST /api/chat                    # Chat general
"""

print("✓ Hook centralizado con gestión completa de estado y API calls")

## 6. Tecnologías y Librerías Utilizadas

### Stack Tecnológico

| Tecnología | Versión | Propósito |
|------------|---------|-----------|
| React | 18.x | Framework UI |
| TypeScript | 5.x | Tipado estático |
| Vite | 5.x | Build tool y dev server |
| TailwindCSS | 3.x | Framework CSS utility-first |
| ESLint | 8.x | Linting y calidad de código |

In [ ]:
# Configuración de desarrollo
"""
Archivos de configuración clave:

1. vite.config.ts
   - Configuración del servidor de desarrollo
   - Proxy para el backend (si es necesario)
   - Optimizaciones de build

2. tsconfig.json
   - Configuración de TypeScript
   - Strict mode habilitado
   - Paths aliases

3. tailwind.config.js
   - Configuración de TailwindCSS
   - Personalización de colores y temas
   - Plugins adicionales

4. package.json - Scripts principales:
   {
     "dev": "vite",                    # Servidor de desarrollo
     "build": "tsc && vite build",     # Build de producción
     "preview": "vite preview",        # Preview del build
     "lint": "eslint ."                # Linting
   }
"""

print("✓ Configuración moderna con Vite y TypeScript")

## 7. Flujo de Datos y Comunicación

### Diagrama de Flujo